# GE_Analysis.ipynb
This code serves to construct a data frame of the last year of Grand Exchange prices from Old School Runescape (OSRS), then will conduct a series of example analyses with said data.

This is an exercise in web scraping with APIs, general coding skills in Python, and data analysis skills in Python using Pandas.

This project would not be possible without the work done by Runelite and the OSRS Wiki!

## Import libraries
These libraries are used in this Jupyter notebook to execute the code contained within. "wikiscraper" is wikiscraper.py, a standalone python script that defines core functions for data scraping and is also packaged with this project. Simply save it in the same directory as this Jupyter notebook and it should work!

In [1]:
import importlib
from pandas.core.interchange.dataframe_protocol import DataFrame

import wikiscraper as ws
# Run if added changes to wikiscraper script
#importlib.reload(ws)
import pandas as pd

## Testing functions
These lines test the core functions of the wikiscraper script. `db` is the mapping of all items in the Grand Exchange (or at least all available entries that are tracked by the OSRS Wiki).

In [24]:
# test getting mapping from API - success
db = pd.json_normalize(ws.get_mapping())

# test getting time series dataframe from single timeseries API call - success
test_ts_1 = pd.json_normalize(ws.get_timeseries(10344, '24h')['data'])

# test getting time series dataframe from multiple timeseries API calls - success
test_ts_2 = ws.get_data([10344, 20011, 12424], '24h', 1)


Using cached mapping
retrieving timeseries for item 10344
retrieving timeseries for item 10344
retrieving timeseries for item 20011
retrieving timeseries for item 12424


## Constructing final data frame
The data frame currently has the following columns:
- `timestamp` *- the Unix timestamp of the record*
- `avgHighPrice` *- the average high price of transactions at the timestamp*
- `avgLowPrice` *- the average low price of transactions at the timestamp*
- `highPriceVolume` *- the number of transactions that have occurred at high price since the last timestamp*
- `lowPriceVolume *- the number of transactions that have occurred at low price since the last timestamp*
- `item_id` *- the OSRS item id*

I would also like to add the following columns from our database mapping:
- `members` *- whether it is a members-only item*
- `limit` *- the 8 hour trade volume limit*
- `lowalch` *- the value of the item at low alch*
- `value` *- honestly might need to look into this - median price? raw value?*
- `highalch` *- the value of the item at high alch*
- `name` *- the in-game name of the item*

In [29]:
test_join = test_ts_2.set_index("item_id").join(db.set_index("id"))

test_join_reordered = test_join[
    ['name'] +
    [col for col in test_join.columns if col != 'name']
]

print(test_join_reordered.head())

                   name   timestamp   avgHighPrice    avgLowPrice  \
item_id                                                             
10344    3rd age amulet  1748476800 97905047.00000 95359578.00000   
10344    3rd age amulet  1748563200 97970500.00000 95088989.00000   
10344    3rd age amulet  1748649600 97842500.00000 93279667.00000   
10344    3rd age amulet  1748736000 95994333.00000 93758672.00000   
10344    3rd age amulet  1748822400 93800000.00000 93449336.00000   

         highPriceVolume  lowPriceVolume  \
item_id                                    
10344                  3               5   
10344                  4              10   
10344                  2               2   
10344                  6               2   
10344                  1               1   

                                                   examine  members  \
item_id                                                               
10344    Fabulously ancient mage protection enchanted i...     Tr

# Apply process to full dataset from OSRS Wiki GE API

In [2]:
# Pull fresh mapping from api
db = pd.json_normalize(ws.get_mapping())

# Create a list of unique item ids
unique_ids = list(db['id'].unique())

# Request API for full listing of ge trade data
full_data = ws.get_data(unique_ids, '24h', 1)

full_data.to_csv('full_data.csv', index=False)

Using cached mapping
retrieving timeseries for item 10344
retrieving timeseries for item 20011
retrieving timeseries for item 12424
retrieving timeseries for item 12437
retrieving timeseries for item 23345
retrieving timeseries for item 23339
retrieving timeseries for item 23336
retrieving timeseries for item 23342
retrieving timeseries for item 28226
retrieving timeseries for item 10350
retrieving timeseries for item 10352
retrieving timeseries for item 12426
retrieving timeseries for item 10342
retrieving timeseries for item 20014
retrieving timeseries for item 10348
retrieving timeseries for item 10346
retrieving timeseries for item 23242
retrieving timeseries for item 10334
retrieving timeseries for item 10332
retrieving timeseries for item 10330
retrieving timeseries for item 10340
retrieving timeseries for item 10338
retrieving timeseries for item 10336
retrieving timeseries for item 12422
retrieving timeseries for item 10392
retrieving timeseries for item 25775
retrieving timese